# qec-bench — Kaggle training run

Trains the per-distance neural decoders on Kaggle's free GPU tier. Enable
**Settings -> Accelerator -> GPU** and **Settings -> Internet -> On**.

Every shell step runs through a helper that (a) raises on a non-zero exit so a
failure shows up as an ERROR status instead of a silently "complete" run, and
(b) tees all output to `/kaggle/working/run.log`, which is always downloadable
even if a later cell fails. The trained checkpoints are the only other thing
written to `/kaggle/working` (the dataset is generated to ephemeral `/tmp` to
keep the output small), so pulling the result back is fast.

In [ ]:
# Config for this run (see configs/ in the repo).
DATASET = "train_v2"
TRAIN_CONFIG = "mlp_v3"

import subprocess, sys, os, pathlib

WORK = "/kaggle/working"
DATA = "/tmp/qec-data"
WEIGHTS = f"{WORK}/weights"
LOG = f"{WORK}/run.log"
os.makedirs(WEIGHTS, exist_ok=True)
pathlib.Path(LOG).write_text("")  # fresh log each run

def sh(cmd):
    print(">>>", cmd, flush=True)
    with open(LOG, "a") as lf:
        lf.write(f"
>>> {cmd}
"); lf.flush()
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); sys.stdout.flush()
            lf.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"FAILED ({p.returncode}): {cmd}")


In [ ]:
sh("nvidia-smi -L")

In [ ]:
# Clone the repo and install qecbench. Kaggle already ships numpy + torch,
# so install only the deps its base image lacks, then the package itself
# without touching the preinstalled torch (avoids a slow/risky reinstall).
if not os.path.isdir("/kaggle/working/qec-bench"):
    sh("git clone --depth 1 https://github.com/Lucas-Maingi/qec-bench.git /kaggle/working/qec-bench")
os.chdir("/kaggle/working/qec-bench")
sh("pip install -q stim pymatching pyyaml")
sh("pip install -q -e . --no-deps")
sh("python -c 'import qecbench, torch; print(\"qecbench\", qecbench.__version__, \"torch\", torch.__version__, \"cuda\", torch.cuda.is_available())')

## 1/2 — Training dataset (generated to /tmp, ~10 min)

In [ ]:
sh(f"qecbench generate --config configs/datasets/{DATASET}.yaml --out {DATA}")

## 2/2 — Train the per-distance decoders on GPU

In [ ]:
sh(f"qecbench train --config configs/train/{TRAIN_CONFIG}.yaml --data {DATA}/{DATASET} --out {WEIGHTS} --device cuda")

## Result — checkpoints land in /kaggle/working/weights

Commit the notebook (*Save & Run All*) so the output persists, then download
the files under `weights/` from the Output tab (a few MB).

In [ ]:
for f in sorted(os.listdir(WEIGHTS)):
    print(f, os.path.getsize(f"{WEIGHTS}/{f}") // 1024, "KB")
print("
--- tail of run.log ---")
print("".join(open(LOG).readlines()[-15:]))